<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

Installo librerie di interesse e copio directory di riferimento da Git-Hub

In [38]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-ita poppler-utils
!pip install pdf2image pytesseract pandas pillow

fatal: destination path 'Lettura_bilanci' already exists and is not an empty directory.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-ita is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 9 not upgraded.


###Definisco i campi da estrarre dai bilanci con le logiche di include e exclude

In [25]:
#definisco valori da includere o escludere nella ricerca del campo specifico
campi_bilancio = {
    "totale_attivo": {
        "include": ["Totale attivo"],
        "exclude": ["circolante"]
    },

    "patrimonio_netto": {
        "include": ["Totale patrimonio netto"],
        "exclude": []
    },

    "totale_attivo_circolante": {
        "include": ["Totale attivo circolante"],
        "exclude": []
    },

    "Totale_disponibilità_liquide": {
        "include": ["Totale disponibilità liquide", "IV - Disponibilità liquide"],
        "exclude": ["esercizio"]
    },

    "Totale_debiti": {
        "include": ["Totale debiti"],
        "exclude": ["verso", "rappresentati", "tributari"]
    },

    "Totale_rimanenze": {
        "include": ["Totale rimanenze", "I - Rimanenze"],
        "exclude": []
    },

    "Totale_crediti_verso_clienti": {
        "include": ["Totale crediti verso clienti", "II - Crediti", "1) Verso clienti"],
        "exclude": []
    },

    "Totale_debiti_verso_fornitori": {
        "include": ["Totale debiti verso fornitori", "Debiti verso fornitori per merci e servizi"],
        "exclude": []
    },

    "Totale_debiti_verso_banche": {
        "include": ["Totale debiti verso banche"],
        "exclude": []
    },

    "avviamento": {
        "include": ["avviamento"],
        "exclude": []
    },

    "ricavi_vendite_e_prestazioni": {
        "include": ["ricavi delle vendite e delle prestazioni"],
        "exclude": []
    },

    "Utile_perdita_esercizio": {
        "include": ["Utile (perdita) dell'esercizio"],
        "exclude": []
    },

    "Risultato_prima_delle_imposte": {
        "include": ["Risultato prima delle imposte"],
        "exclude": []
    },

    "Totale_ammortamenti_e_svalutazioni": {
        "include": ["Totale ammortamenti e svalutazioni"],
        "exclude": []
    },

    "Totale_interessi_e_altri_oneri": {
        "include": ["Totale interessi e altri oneri finanziari", "Totale interessi ed altri oneri finanziari"],
        "exclude": []
    },

    "immobilizzazioni_immateriali": {
        "include": ["Totale immobilizzazioni immateriali", "I - Immobilizzazioni immateriali"],
        "exclude": []
    },

    "immobilizzazioni_materiali": {
        "include": ["Totale immobilizzazioni materiali", "II - Immobilizzazioni materiali"],
        "exclude": []
    },

    "immobilizzazioni_finanziarie": {
        "include": ["Totale immobilizzazioni finanziarie", "III - Immobilizzazioni finanziarie"],
        "exclude": []
    },

    "Flusso_finanziario_attivita_operativa": {
        "include": ["Flusso finanziario dell'attività operativa"],
        "exclude": []
    },

    "Flusso_finanziario_attività_investimento": {
        "include": ["Flusso finanziario dell'attività di investimento", "Totale flusso finanziario dell'attività di investimento"],
        "exclude": []
    },

    "Flusso_finanziario_attività_finanziamento": {
        "include": ["Flusso finanziario dell'attività di finanziamento", "Totale flusso finanziario dell'attività di finanziamento"],
        "exclude": []
    },

  }

###Definisco keywords

In [26]:
#definisco keywords per identificare prima pagina tabella OIC
keywords_first_page = ["immobilizzazioni immateriali", "crediti",
            "immobilizzazioni materiali", "immobilizzazioni finanziarie"]

iniz_deb = "d) debiti"
fin_deb = "e) ratei e risconti"
alias_correnti = ["esigibili entro l'esercizio successivo",
                  "sigibili entro l'esercizio successivo"]


###Definisco funzioni utili per le analisi delle stringhe

In [27]:
import unicodedata
def norm(x):
    """
    pulisce la stringa passata in argomento, effettuando:
    - conversione a stringa
    - trasforma in minuscolo
    - sostituisce a capo con spazio
    - sostituisco con un unico tipo di apostrofo o trattino
    - trasformo in generale in forma standardizzata NFKC
    - rimuove eventuali spazi multipli
    """
    x = str(x).lower()
    x = x.replace("\n", " ")
    x = x.replace("’", "'").replace("‘", "'").replace("`", "'")
    x = x.replace("–", "-").replace("—", "-")
    x = unicodedata.normalize("NFKC", x)
    return " ".join(x.split())

def riga_valida(valori):
    """
    verifica che nella riga valori sia presente almeno un valore non nullo
    """
    return any(pd.notna(v) and str(v).strip() != "" for v in valori)

def pulisci_numero(x):
    """
    prova a convertire un valore letto in numero float, effettuando:
    - se nullo ritorna nan
    - conversione a stringa
    - se inizia con parentesi lo interpreta come negativo
    - interpreta "." come separatore migliaia e "," per i decimali; converte a
      formato inglese
    - trasforma in float
    """
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x in ("", "-"):
        return np.nan

    negativo = x.startswith("(") and x.endswith(")")
    x = x.replace("(", "").replace(")", "").strip()

    if "," in x:
        x = x.replace(".", "").replace(",", ".")
    else:
        x = x.replace(".", "")

    try:
        val = float(x)
        return -val if negativo else val
    except:
        return np.nan


def match_testo(testo, include, exclude):
    """
    verifica che nel testo passato in argomento sia presente la stringa di include
    e non sia presente nessuna delle stringhe passate in exclude (include è singola)
    """
    testo = norm(testo)
    return include.lower() in testo and not any(e.lower() in testo for e in exclude)


def calcola_passivita_correnti(tabella, col_label, cols_valori,
                               iniz_deb, fin_deb, alias_correnti):
    """
    Calcola le passività correnti, in argomento vengono passati:
    - tabella: un DataFrame pandas.
    - col_label: indice della colonna che contiene le etichette testuali.
    - cols_valori: lista delle colonne che contengono i valori numerici da sommare.
    - eventuale inizio e fine del blocco debiti e alias

    l'algoritmo effettua i seguenti passaggi:
    - prende la colonna delle etichette e normalizza il testo
    - identifica inizio e fine del blocco debiti
    """
    testi = tabella.iloc[:, col_label].astype(str).apply(norm)

    # inizio/fine blocco debiti
    idx_inizio = testi[testi.str.contains(iniz_deb, na=False, regex=False)].index
    idx_fine = testi[testi.str.contains(fin_deb, na=False, regex=False)].index

    #ritorna none se non trova il blocco iniziale o finale
    if len(idx_inizio) == 0 or len(idx_fine) == 0:
        return [None] * len(cols_valori)

    #prende il primo elemento trovato corrispondente all'inizio blocco e come fine
    #il primo elemento della fine successivo all'elemento precedente
    idx_inizio = idx_inizio[0]
    idx_fine = next((i for i in idx_fine if i > idx_inizio), None)

    #se non c'è riga successiva all'inizio ritorna none
    if idx_fine is None:
        return [None] * len(cols_valori)

    #estrae la sottotabella interessata, tenendo solo la colonna delle etichette
    #e la colonna con i valori numerici da sommare; normalizza le etichette
    sotto = tabella.loc[idx_inizio:idx_fine, [col_label] + cols_valori].copy()
    sotto["label_norm"] = sotto.iloc[:, col_label].astype(str).apply(norm)

    #verifica se la riga contiene uno degli alias definiti e copia tali valori in
    #dataset a parte
    mask = sotto["label_norm"].apply(lambda x: any(a in x for a in alias_correnti))
    righe_correnti = sotto.loc[mask].copy()

    #se non trova righe restituisce none
    if righe_correnti.empty:
        return [None] * len(cols_valori)

    #pulisce i numeri presenti nelle righe
    for col in cols_valori:
        righe_correnti[col] = righe_correnti[col].apply(pulisci_numero)

    #calcola la somma e la restituisce
    somme = righe_correnti[cols_valori].sum(axis=0, skipna=True).tolist()
    return [None] * len(cols_valori) if all(pd.isna(v) for v in somme) else somme


###Definisco funzioni per la lettura OCR dei documenti

In [47]:
import re
import pandas as pd
import pytesseract
from pdf2image import convert_from_path


def normalizza_testo(s):
    return " ".join(str(s).lower().split())


def ocr_data(img, lang="ita", psm=6):
    return pytesseract.image_to_data(
        img,
        lang=lang,
        output_type=pytesseract.Output.DICT,
        config=f"--psm {psm}"
    )


def estrai_testo_pagina(img, lang="ita"):
    testo = pytesseract.image_to_string(img, lang=lang, config="--psm 6")
    return normalizza_testo(testo)


def trova_pagina_iniziale(immagini, start_marker, lang="ita"):
    for i, img in enumerate(immagini):
        testo = estrai_testo_pagina(img, lang=lang)
        if start_marker in testo:
            return i
    return None


def estrai_date_e_posizioni(img, lang="ita"):
    data = ocr_data(img, lang=lang, psm=6)
    pattern = re.compile(r"\d{2}[-/]\d{2}[-/]\d{4}")

    trovate = []
    for i, txt in enumerate(data["text"]):
        txt = txt.strip()
        if pattern.fullmatch(txt):
            left = data["left"][i]
            top = data["top"][i]
            width = data["width"][i]
            height = data["height"][i]
            trovate.append({
                "text": txt,
                "left": left,
                "top": top,
                "right": left + width,
                "bottom": top + height,
                "x_center": left + width / 2
            })

    if len(trovate) < 2:
        return None

    trovate = sorted(trovate, key=lambda x: x["x_center"])
    return trovate[-2], trovate[-1]


def estrai_righe_area(img, lang="ita", psm=4):
    data = ocr_data(img, lang=lang, psm=psm)

    parole = []
    n = len(data["text"])
    for i in range(n):
        txt = data["text"][i].strip()
        if not txt:
            continue

        try:
            conf = float(data["conf"][i])
        except:
            conf = -1

        if conf < 20:
            continue

        parole.append({
            "text": txt,
            "left": data["left"][i],
            "top": data["top"][i],
            "width": data["width"][i],
            "height": data["height"][i]
        })

    if not parole:
        return []

    parole = sorted(parole, key=lambda x: (x["top"], x["left"]))

    righe = []
    corrente = [parole[0]]
    soglia_y = 10

    for p in parole[1:]:
        top_medio = sum(x["top"] for x in corrente) / len(corrente)
        if abs(p["top"] - top_medio) <= soglia_y:
            corrente.append(p)
        else:
            righe.append(corrente)
            corrente = [p]
    righe.append(corrente)

    risultato = []
    for r in righe:
        r = sorted(r, key=lambda x: x["left"])
        testo = " ".join(x["text"] for x in r).strip()
        top_medio = sum(x["top"] for x in r) / len(r)
        risultato.append({
            "text": testo,
            "top": top_medio
        })

    return risultato


def unisci_righe_per_top(label_rows, d1_rows, d2_rows, tolleranza=12):
    tutte_le_top = []

    for r in label_rows:
        tutte_le_top.append(r["top"])
    for r in d1_rows:
        tutte_le_top.append(r["top"])
    for r in d2_rows:
        tutte_le_top.append(r["top"])

    if not tutte_le_top:
        return []

    tutte_le_top = sorted(tutte_le_top)

    cluster = []
    gruppo = [tutte_le_top[0]]
    for t in tutte_le_top[1:]:
        if abs(t - sum(gruppo) / len(gruppo)) <= tolleranza:
            gruppo.append(t)
        else:
            cluster.append(sum(gruppo) / len(gruppo))
            gruppo = [t]
    cluster.append(sum(gruppo) / len(gruppo))

    def closest_text(rows, y, tol):
        candidati = [r for r in rows if abs(r["top"] - y) <= tol]
        if not candidati:
            return None
        candidati = sorted(candidati, key=lambda r: abs(r["top"] - y))
        return candidati[0]["text"]

    out = []
    for y in cluster:
        label = closest_text(label_rows, y, tolleranza)
        data1 = closest_text(d1_rows, y, tolleranza)
        data2 = closest_text(d2_rows, y, tolleranza)
        if label or data1 or data2:
            out.append([label, data1, data2])

    return out


def riga_spuria(label):
    if not label:
        return False

    l = normalizza_testo(label)
    patterns = [
        "bilancio di esercizio al",
        "generato automaticamente",
        "conforme alla tassonomia",
        "pag.",
        "gollinucci srl",
        "soggetta a direzione e coordinamento",
        "v.2."
    ]
    return any(p in l for p in patterns)


def estrai_righe_tabella_da_pagina(img, colonne, lang="ita"):
    w, h = img.size

    x1 = colonne["x1"]
    x2 = colonne["x2"]
    y_start = colonne["y_start"]

    area_label = img.crop((0, y_start, x1, h))
    area_d1 = img.crop((x1, y_start, x2, h))
    area_d2 = img.crop((x2, y_start, w, h))

    righe_label = estrai_righe_area(area_label, lang=lang, psm=4)
    righe_d1 = estrai_righe_area(area_d1, lang=lang, psm=4)
    righe_d2 = estrai_righe_area(area_d2, lang=lang, psm=4)

    righe = unisci_righe_per_top(righe_label, righe_d1, righe_d2, tolleranza=12)
    return righe


def trova_tabella_oic_multipaagina(
    pdf_path,
    start_marker="stato patrimoniale",
    end_marker="conto economico",
    lang="ita",
    dpi=300
):
    immagini = convert_from_path(pdf_path, dpi=dpi)

    start_marker = normalizza_testo(start_marker)
    end_marker = normalizza_testo(end_marker)

    pagina_start = trova_pagina_iniziale(immagini, start_marker, lang=lang)
    if pagina_start is None:
        return None

    # trova coordinate colonne sulla prima pagina utile
    date_pos = estrai_date_e_posizioni(immagini[pagina_start], lang=lang)
    if date_pos is None:
        return None

    d1, d2 = date_pos

    colonne = {
        "x1": int(d1["left"] - 40),
        "x2": int(d2["left"] - 40),
        "y_start": int(min(d1["top"], d2["top"]) + 25)
    }

    risultati = []
    collecting = False

    for i in range(pagina_start, len(immagini)):
        img = immagini[i]

        righe = estrai_righe_tabella_da_pagina(img, colonne, lang=lang)

        for label, data1, data2 in righe:
            label_norm = normalizza_testo(label or "")

            if not collecting:
                if start_marker in label_norm:
                    collecting = True
                continue

            if label and end_marker in label_norm:
                return pd.DataFrame(risultati, columns=["label", "data1", "data2"])

            if riga_spuria(label):
                continue

            if label is None and data1 is None and data2 is None:
                continue

            risultati.append([label, data1, data2])

    if risultati:
        return pd.DataFrame(risultati, columns=["label", "data1", "data2"])

    return None

Definisco funzione per leggere il pdf dalla cartella ed identificare la tabella OIC con i valori numerici di bilancio

In [28]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords):
    """
    trova e restituisce la tabella bilancio OIC, in argomento:
    - pdf_path: indica il percorso del file pdf da analizzare
    - keywords indica le parole chiave che identificano la prima pagina del bilancio
    """
    #leggo il pdf ed estraggo le tabelle; per ogni tabella la converto in un unica
    #stringa e controllo che le keywords siano presenti al suo interno; in caso
    #positivo salvo l'indice della pagina di riferimento.
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        #legge a partire dalla tabella iniziale di riferimento, continua a leggere
        #finchè sono presenti tabelle nelle pagine successive, unendo tutto il
        #contenuto in un dataframe pandas
        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


Definisco funzione che estrae i valori di interesse dalla tabella numerica OIC

In [29]:
import pandas as pd
import numpy as np

def summary_tabella(tabella, labels_dict):
    """
    analizza la tabella oic, restituendo i valori di interesse indicati in labels_dict
    """
    #gestisco in modo diverso se la tabella ha 5 colonne o 3, se ne ha 5 unisco
    #le prime due colonne
    if tabella.shape[1] == 3:
        date = tabella.iloc[0, 1:3].tolist()

    elif tabella.shape[1] == 5:
        tabella.iloc[:,0] = (
            tabella.iloc[:,0].fillna("").astype(str).str.strip()
                .str.cat(tabella.iloc[:,1].fillna("").astype(str).str.strip(),
                            sep=" ")
        )
        # elimina la colonna 1
        tabella.drop(tabella.columns[1], axis=1, inplace=True)

        # ricavo le date sui nuovi indici 1 e 2
        date = tabella.iloc[0, 1:3].tolist()

    else:
        return pd.DataFrame()

    col_label = tabella.columns[0]
    cols_valori = tabella.columns[1:3].tolist()

    righe = {}
    col_testi = tabella.iloc[:, col_label].astype(str)

    #scorre sui nomi da estrarre, applicando le logiche di include ed exclude
    for nome_output, regola in labels_dict.items():
        trovato = False

        for include in regola["include"]:
            mask = col_testi.apply(lambda x: match_testo(x, include, regola["exclude"]))
            if not mask.any():
                continue

            for _, row in tabella.loc[mask, cols_valori].iterrows():
                valori = row.tolist()
                if riga_valida(valori):
                    righe[nome_output] = [pulisci_numero(v) for v in valori]
                    trovato = True
                    break

            if trovato:
                break

        if not trovato:
            righe[nome_output] = [None] * len(cols_valori)

    righe["passivita_correnti"] = calcola_passivita_correnti(tabella, col_label, cols_valori,
                                                             iniz_deb, fin_deb, alias_correnti)

    return pd.DataFrame.from_dict(righe, orient="index", columns=date)



Leggo tutti i pdf nella cartella e stampo la tabella di sintesi

In [30]:
import glob
import os

pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path, keywords_first_page)

    if tabella is not None:
        summary[nome] = summary_tabella(tabella, campi_bilancio)



# stampa i risultati ottenuti
for nome, df in summary.items():
    print("\n" + "=" * 80)
    print(f"BILANCIO: {nome}")
    print("=" * 80)
    print(df)


BILANCIO: Gollinucci - 2024
                                           31-12-2024  31-12-2023
totale_attivo                              34087270.0  22766880.0
patrimonio_netto                           19350241.0  15153957.0
totale_attivo_circolante                   11717234.0  18259510.0
Totale_disponibilità_liquide                4709597.0  11981963.0
Totale_debiti                              13232032.0   6810402.0
Totale_rimanenze                            3079966.0   2829415.0
Totale_crediti_verso_clienti                2624411.0   3145372.0
Totale_debiti_verso_fornitori               2655387.0   3077701.0
Totale_debiti_verso_banche                  9579023.0   2871256.0
avviamento                                 16032816.0         0.0
ricavi_vendite_e_prestazioni               19674840.0  20055260.0
Utile_perdita_esercizio                      642760.0   2172984.0
Risultato_prima_delle_imposte               1648222.0   2840912.0
Totale_ammortamenti_e_svalutazioni          346

Debug

In [32]:

pdf_path= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
#pdf_path= "Lettura_bilanci/deposito bilanci/Gollinucci - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path, keywords_first_page)
summary_tabella(tabella, campi_bilancio)


,30 giugno 2025,30 giugno 2024
totale_attivo,932834011.0,879915161.0
patrimonio_netto,305929558.0,300611685.0
totale_attivo_circolante,439099320.0,416207934.0
Totale_disponibilità_liquide,127609819.0,106847278.0
Totale_debiti,591475140.0,545013151.0
Totale_rimanenze,205612371.0,201996264.0
Totale_crediti_verso_clienti,55501954.0,60932295.0
Totale_debiti_verso_fornitori,236897353.0,228044458.0
Totale_debiti_verso_banche,191590518.0,168282068.0
avviamento,NaN,NaN


In [48]:
df = trova_tabella_oic_multipaagina(
    pdf_path="Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf",
    start_marker="stato patrimoniale",
    end_marker="conto economico",
    lang="ita",
    dpi=300
)

df.head(50)
df.tail(50)

,label,data1,data2
76,Varie altre riserve,51,(2.879)
77,Totale altre riserve,64.138.921,17.230.521
78,VII Riserva per operazioni di copertura dei fl...,(381.859),381.860
79,-,None,None
80,VIII Utili (perdite) portati a nuovo,(16.360),(871.643)
81,-,None,None
82,IX Utile (perdita) dell'esercizio,(8.901.374),1.648.856
83,-,None,None
84,X Riserva negativa per azioni proprie in porta...,0,0
85,-,None,None


In [49]:
print(df)

                                                 label        data1  \
0                                               Attivo         None   
1    A) Crediti verso soci per versamenti ancora do...         None   
2    Totale crediti verso soci per versamenti ancor...         None   
3                                  B) Immobilizzazioni         None   
4                       | Immobilizzazioni immateriali         None   
..                                                 ...          ...   
121                                      Totale debiti   87.945.470   
122                                E) Ratei e risconti    8.575.634   
123                                     Totale passivo  160.159.167   
124                                              7 di.           73   
125                                                  -         None   

           data2  
0           None  
1           None  
2           None  
3           None  
4           None  
..           ...  
121   66.838.1